# Semantic Memory | Agent Memory System

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import Dict, List, Tuple
from dataclasses import dataclass, field

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
# Semantic Memory: Knowledge Graph + Vector Store for Semantic Retrieval

@dataclass
class Entity:
    name: str
    entity_type: str
    properties: Dict[str, str] = field(default_factory=dict)

class SemanticMemory:
    """Combines a knowledge graph (structured facts) with a vector store (semantic search)."""

    def __init__(self):
        self.entities: Dict[str, Entity] = {}
        self.relations: List[Tuple[str, str, str]] = []  # (subject, predicate, object)
        self.vector_store = InMemoryVectorStore(embeddings)

    def add_entity(self, name: str, entity_type: str, **properties):
        self.entities[name] = Entity(name=name, entity_type=entity_type, properties=properties)
        # Also index in vector store for semantic retrieval
        props_text = ", ".join(f"{k}={v}" for k, v in properties.items())
        doc = Document(page_content=f"{name} is a {entity_type}. {props_text}",
                       metadata={"entity": name, "type": entity_type})
        self.vector_store.add_documents([doc])

    def add_relation(self, subject: str, predicate: str, obj: str):
        self.relations.append((subject, predicate, obj))
        # Index the relationship for semantic search
        doc = Document(page_content=f"{subject} {predicate} {obj}",
                       metadata={"subject": subject, "predicate": predicate, "object": obj})
        self.vector_store.add_documents([doc])

    def query_graph(self, entity_name: str) -> str:
        """Exact lookup: get all facts about a specific entity."""
        entity = self.entities.get(entity_name)
        if not entity:
            return f"No knowledge about '{entity_name}'."
        lines = [f"{entity_name} ({entity.entity_type})"]
        for k, v in entity.properties.items():
            lines.append(f"  {k}: {v}")
        for s, p, o in self.relations:
            if s == entity_name:
                lines.append(f"  {p} -> {o}")
            elif o == entity_name:
                lines.append(f"  <- {p} {s}")
        return "\n".join(lines)

    def query_semantic(self, question: str, k: int = 3) -> str:
        """Semantic search: find facts related to a natural language question."""
        results = self.vector_store.similarity_search(question, k=k)
        return "\n".join(f"- {r.page_content}" for r in results) if results else "No relevant knowledge found."

    def get_context(self, entity_name: str = None, question: str = None) -> str:
        """Combined retrieval: graph lookup + semantic search."""
        parts = []
        if entity_name:
            parts.append(f"[Graph Lookup]\n{self.query_graph(entity_name)}")
        if question:
            parts.append(f"[Semantic Search]\n{self.query_semantic(question)}")
        return "\n\n".join(parts)

In [4]:
# Build knowledge base
mem = SemanticMemory()
mem.add_entity("Alice", "customer", plan="enterprise", industry="fintech", joined="2023-01")
mem.add_entity("Project Alpha", "project", status="active", budget="500K", tech_stack="Python, FastAPI")
mem.add_entity("Bob", "customer", plan="startup", industry="healthcare", joined="2024-06")
mem.add_relation("Alice", "owns", "Project Alpha")
mem.add_relation("Alice", "prefers", "email communication")
mem.add_relation("Project Alpha", "uses", "API v2")
mem.add_relation("Bob", "interested_in", "API integration")

# Demo 1: Exact graph lookup for known entity
print("[Query: Who is Alice?]")
print(mem.query_graph("Alice"))

# Demo 2: Semantic search for open-ended question (no entity name needed)
print("\n[Query: Who needs API help?]")
print(mem.query_semantic("API integration upgrade"))

# Demo 3: Combined retrieval powering an agent response
context = mem.get_context(entity_name="Alice", question="API upgrade discussion")
response = model.invoke(
    f"You are a customer success agent with this knowledge:\n\n{context}\n\n"
    f"Customer asks: 'Can we schedule a call to discuss upgrading our API integration?'"
)
print(f"\n[Agent Response]\n{response.content}")

[Query: Who is Alice?]
Alice (customer)
  plan: enterprise
  industry: fintech
  joined: 2023-01
  owns -> Project Alpha
  prefers -> email communication

[Query: Who needs API help?]
- Bob interested_in API integration
- Project Alpha uses API v2
- Project Alpha is a project. status=active, budget=500K, tech_stack=Python, FastAPI

[Agent Response]
Hi Alice,

Thanks for reaching out! Since you prefer email communication, I'll provide some information here and we can proceed with scheduling a call if needed.

I understand you're interested in discussing an upgrade to your API integration for Project Alpha. Currently, Project Alpha is using API v2, which is quite robust, but if you're looking into possible improvements or exploring new features, an upgrade could be a great idea.

To ensure we cover all aspects effectively, could you let me know a bit more about what you're hoping to achieve with the upgrade? This will help us prepare and make the most of our discussion.

If you’d like to